# Set Snowflake Connection Properties with Public/Private Key Pair
Configure this machine for easy connection to a Snowflake training account provided to you by the instructor using key pair authentication.

Steps:
1. Generate public/private key pair; set the proper privileges on generated files.
2. Get Snowflake connection details (account, username, password).
3. Log in to Snowflake and configure the user:
   - Configure the Snowflake user with the public key previously generated.
   - Set user's initial objects and default context.
4. Save the Snowflake login credentials (account, user, private key file) to a local config file.

Subsequent notebooks will connect to Snowflake using the configuration set up here, to verify correctness.

In [2]:
import os
import getpass
from urllib.parse import quote

# Load Jupyter/IPython sql magic
%load_ext sql

# config_dir = '/home/jovyan/.ssh'
config_dir = '/Users/richardkirk/.ssh'
sf_configfile = config_dir + '/sf_config'

ModuleNotFoundError: No module named 'sql'

### 1. Generate public/private key pair

In [2]:
# Create directory for config files

# TODO don't want to be deleting ssh folder!!
# !rm -rf {config_dir}
# !mkdir {config_dir}

# Produce private key file
!openssl genrsa -out {config_dir}/rsa_key.pem 2048

# Format private key
!cat {config_dir}/rsa_key.pem \
    | egrep -v '\-\-\-' \
    | tr -d '\n' \
    > {config_dir}/rsa_key.pem.trimmed

# Generate formatted public key
!openssl rsa -in {config_dir}/rsa_key.pem -pubout \
    | egrep -v '\-\-\-' \
    | tr -d '\n' \
    > {config_dir}/rsa_key.pub

# Set permissions
!chmod 700 {config_dir}
!chmod 600 {config_dir}/rsa_key.pem*
!chmod 644 {config_dir}/rsa_key.pub

# Set a variable to the public key; view the key
pub_key = open(config_dir + '/rsa_key.pub','r').read()
pub_key

writing RSA key


'MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAylKPMGn/kXB/2IE6V9Ky7XpqjGWVKaGK8Lf1QY/EqX3ny4p+S9IqIpcmHGouQ5MOWsne30RytIsO01iMufN2ckr8w3ymngjSh5nx/vsZrP5rYBYRpeyVd50YVcua1Aum5OgYuNURVGENLoWrFXaSrO7pn4Hyq7QF2rgHE/V7BKhyJ/tHLqJ927iOTDq3Sbj/Mj/8srIpwJqxUkn4ZqgsVnifZgtoNqRVGpPjOwQdLG44/Ahg7f0vgRxynmSvx1AW8HsU5/QsMCCKjlhJeYDDkKG35zjp0w50/ic6n2tuotWjZxWhxP1nw9TSnu3WVCdtx+4kdxA8oTAc/LQjuplk9wIDAQAB'

### 2. Get Snowflake connection details

In [5]:
sf_account     = input('Snowflake Account: ')
sf_user        = input('Snowflake User: ')
sf_password    = getpass.getpass('Snowflake Password: ')

### 3. Log in to Snowflake and configure the user

#### 3.1 Set the user's public key

In [1]:
# Connect to Snowflake
import snowflake.connector

# sf_account = 'cob70986.us-east-1'
# sf_user = 'RKIRK'
# get password from vault


con = snowflake.connector.connect(
    user=sf_user,
    password=sf_password,
    account=sf_account
)
cur = con.cursor()

NameError: name 'sf_user' is not defined

In [7]:
# Install public key in the user's Snowflake properties
cur.execute("alter session set query_tag = 'RSA Key assignment'")
cur.execute("use role securityadmin")
cur.execute("alter user " + sf_user +
            " set RSA_PUBLIC_KEY = '" + pub_key + "'")

#### 3.2 Set initial user objects and default context

In [9]:
user_db        = f"{sf_user}_DB"
user_namespace = f"{user_db}.PUBLIC"
user_wh        = f"{sf_user}_WH"

cur.execute("alter session set query_tag='Set user default context'")
cur.execute("use role ML_MODEL_ROLE")

# cur.execute("create database if not exists " + user_db)
# cur.execute("create warehouse if not exists " + user_wh +
#             " auto_suspend = 180 initially_suspended=true")
# cur.execute("alter user " + sf_user + 
#             " set default_role = training_role " +
#             "     default_warehouse = " + user_wh + 
#             "     default_namespace = " + user_namespace)

cur.execute("alter session unset query_tag")

### 4. Save login details to config file

In [10]:
url = 'https://' + sf_account + '.snowflakecomputing.com'
private_key_file = config_dir + '/rsa_key.pem'

with open(sf_configfile, 'w') as f:
    f.write('ACCOUNT=' + sf_account.upper() + '\n')
    f.write('URL=' + url + '\n')
    f.write('USER=' + sf_user.upper() + '\n')
    f.write('PRIVATE_KEY_FILE=' + private_key_file + '\n')
!chmod 600 {sf_configfile}